In [ ]:
import os
import pandas as pd
from datasets import Dataset

from utils.bq import get_client, query_to_df
from utils.hub_card import push_dataset_card
from utils.reduce_cohort_df import null_triage_outliers, null_pain_column

client, PROJECT_NAME = get_client()

In [ ]:
# Triage information from ed.triage — static, one row per ED visit.
# Chief complaint, ESI acuity level, and initial vitals recorded at triage.
# One-to-one with edstays, so row count should match cohort size.
# Used as baseline/static state features (acuity, chiefcomplaint) and initial vital signs.

query_triage = f"""
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `{PROJECT_NAME}.rl_project.cohort_base`
)
SELECT
  t.stay_id   AS ed_stay_id,
  t.subject_id,
  t.temperature,
  t.heartrate,
  t.resprate,
  t.o2sat,
  t.sbp,
  t.dbp,
  t.pain,
  t.acuity,
  t.chiefcomplaint
FROM `physionet-data.mimiciv_ed.triage` t
INNER JOIN cohort_subjects cs ON t.stay_id = cs.ed_stay_id
ORDER BY t.stay_id
"""

print("Running triage query...")
df_triage = query_to_df(client, query_triage)
print(f"Shape: {df_triage.shape}")
print(f"Triage rows vs expected cohort size: {len(df_triage):,}  (should be ~399,573)")
df_triage.head()

In [ ]:
DESCRIPTION = (
    "ED triage records from ed.triage. One row per ED visit. Includes ESI acuity level, "
    "chief complaint, and initial vital signs at triage. Includes back-to-back ED visits "
    "that are handled when merging with the cohort."
)

ds = Dataset.from_pandas(df_triage, split='triage')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="triage", data_dir="triage")
push_dataset_card("ADS599-Capstone/raw_data", config_name="triage", split="triage", data_dir="triage", description=DESCRIPTION)
print("Pushed triage to HuggingFace Hub.")

In [4]:
from huggingface_hub import DatasetCard


card = DatasetCard.load("ADS599-Capstone/raw_data")

card.data['configs']

[{'config_name': 'cohort_base',
  'data_files': [{'split': 'base', 'path': 'cohort/base-*'}]},
 {'config_name': 'cohort_with_triage',
  'data_files': [{'split': 'with_triage', 'path': 'cohort/with_triage-*'}]},
 {'config_name': 'ecg', 'data_files': [{'split': 'ecg', 'path': 'ecg/ecg-*'}]},
 {'config_name': 'ed_only_meds',
  'data_files': [{'split': 'ed_only', 'path': 'stay_medications/ed_only-*'}]},
 {'config_name': 'height',
  'data_files': [{'split': 'height', 'path': 'height/height-*'}]},
 {'config_name': 'labs_base',
  'data_files': [{'split': 'train', 'path': 'lab_events/train-*'}]},
 {'config_name': 'medrecon',
  'data_files': [{'split': 'medrecon',
    'path': 'pre_arrival_meds/medrecon-*'}]},
 {'config_name': 'meds_admitted',
  'data_files': [{'split': 'meds_admit',
    'path': 'stay_medications/meds_admit-*'}]},
 {'config_name': 'microbiology_events',
  'data_files': [{'split': 'microbiology_events',
    'path': 'microbiology/microbiology_events-*'}]},
 {'config_name': 'radiol

In [7]:
from datasets import load_dataset

# Load the processed cohort (post AGAINST ADVICE removal, consecutive visit collapse,
# stay window and boarding time columns) and merge triage in.
# cohort_df = load_dataset("ADS599-Capstone/raw_data", name="cohort_full", split="cohort_full").to_pandas()

cohort_with_triage = cohort_df.merge(df_triage, on=['ed_stay_id', 'subject_id'], how='inner')
print(f"cohort rows: {len(cohort_df):,}")
print(f"triage rows: {len(df_triage):,}")
print(f"merged rows: {len(cohort_with_triage):,}")
print(f"unique ed_stay_id: {cohort_with_triage['ed_stay_id'].nunique():,}")
cohort_with_triage.head()

cohort rows: 397,675
triage rows: 399,573
merged rows: 397,675
unique ed_stay_id: 397,675


,subject_id,ed_stay_id,hadm_id,ed_intime,ed_outtime,disposition,race,arrival_transport,first_careunit,first_icu_intime,...,ed_boarding_time_hrs,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint
0,10002013,34931809,21763296.0,2165-11-22 20:54:00,2165-11-23 08:20:42,ADMITTED,WHITE,WALK IN,Medical Intensive Care Unit (MICU),2165-11-23 08:20:42,...,0.028333,98.400000000,108.000000000,16.000000000,98.000000000,160.000000000,70.000000000,0,2.000000000,"Chest pain, Wound eval"
1,10014354,38849383,27562275.0,2148-07-17 20:36:00,2148-07-18 04:46:00,ADMITTED,WHITE,AMBULANCE,Cardiology Surgery Intermediate,NaT,...,2.250000,97.700000000,98.000000000,16.000000000,96.000000000,105.000000000,70.000000000,10,2.000000000,"Chest pain, Nausea"
2,10018862,35211473,22949345.0,2149-05-01 17:05:00,2149-05-01 22:22:00,ADMITTED,WHITE,WALK IN,Transplant,NaT,...,0.650000,99.300000000,90.000000000,20.000000000,100.000000000,120.000000000,66.000000000,7,3.000000000,"Abd pain, N/V"
3,10019003,38020791,26529390.0,2155-05-17 21:03:00,2155-05-18 00:03:15,ADMITTED,WHITE,WALK IN,Hematology/Oncology Intermediate,NaT,...,24.020833,98.100000000,97.000000000,16.000000000,93.000000000,117.000000000,60.000000000,5,2.000000000,"Abnormal CT, LUQ abd pain"
4,10021395,31017572,21372240.0,2132-07-30 18:58:00,2132-07-31 04:56:00,ADMITTED,WHITE - OTHER EUROPEAN,WALK IN,Medicine,NaT,...,1.466667,97.000000000,61.000000000,18.000000000,100.000000000,102.000000000,60.000000000,7,2.000000000,Dizziness


# Removal of NA's

In [9]:
# EDA showed triage cols were about 5-10% missing, so dropping those only removes about 10% of the total number of unique ED stays, more than
# enough for robust analysis

triage_cols = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity']
cohort_with_triage.dropna(subset=triage_cols, inplace=True)

In [10]:
# Removing outliers that are outside of plausible ranges reduces the cohort count by another 20k

cohort_with_triage = null_pain_column(cohort_with_triage)
cohort_with_triage = null_triage_outliers(cohort_with_triage)

cohort_with_triage.dropna(subset=triage_cols, inplace = True)

# Hugging Face Push

In [ ]:
WITH_TRIAGE_DESCRIPTION = (
    "Primary cohort table with ED triage data merged in. One row per ED visit. "
    "Built by merging the processed cohort (AGAINST ADVICE removed, consecutive ED visits "
    "collapsed, stay_window and ed_boarding_time_hrs added) with ed.triage on ed_stay_id + subject_id. "
    "Triage rows for the second visit in a consecutive pair are dropped by the inner join. "
    "Intended as the main feature engineering input for the RL model."
)

ds_with_triage = Dataset.from_pandas(cohort_with_triage, split='with_triage')
ds_with_triage.push_to_hub("ADS599-Capstone/raw_data", config_name="with_triage", data_dir="cohort")
push_dataset_card("ADS599-Capstone/raw_data", config_name="with_triage", split="with_triage", data_dir="cohort", description=WITH_TRIAGE_DESCRIPTION)
print("Pushed with_triage to HuggingFace Hub.")